# Colab：Ollama + Nemotron 3.5 Bolt + ngrok

在 VS Code 開啟本檔，選擇 **Select Kernel → Colab**，再從上到下逐格執行。第 6 步會要求輸入 ngrok Authtoken。

**流程：** 安裝 Ollama → 啟動服務 → 下載／測試模型 → 安裝 ngrok → 設定 Authtoken → 寫入 Traffic Policy → 啟動 ngrok → 測試遠端網址。

- 模型：`nemotron-3.5-bolt:e4b`
- Context：`65536`；Keep Alive：`-1`；OLLAMA_ORIGINS：`*`
- Traffic Policy：只修改 `host: localhost`，不加入帳號密碼或其他模式。
- 安裝與模型測試以 Notebook 直接顯示指令輸出；背景服務若失敗，顯示最近的實際錯誤紀錄。

**注意：** ngrok Authtoken 用於啟用 ngrok 服務，**不是**外部使用者的 API 存取密碼。此版本公開端點未設定存取驗證，請勿公開分享網址；使用完請停止 ngrok。Colab 執行階段可能中斷，也應遵守 Colab 的服務使用規定。

## 1. 安裝 Ollama

以官方安裝腳本安裝 Ollama，指令輸出會直接顯示在儲存格下方。若安裝失敗，這一格就會停止，不會誤認為成功。

In [ ]:
%%bash
set -euo pipefail
# 安裝 Ollama 所需工具。
apt-get update -qq
apt-get install -y zstd curl
# 執行 Ollama 官方安裝腳本。
curl -fsSL https://ollama.com/install.sh | sh
# 顯示安裝版本。
ollama --version

## 2. 設定環境變數並啟動 Ollama

為了讓 `OLLAMA_CONTEXT_LENGTH` 和 `OLLAMA_KEEP_ALIVE` **真正套用到服務**，使用 Python 啟動 Ollama，並將環境變數傳入新程序。這一格會重新啟動 Colab 執行階段內的 Ollama；啟動後確認本機 API 可以存取。

In [ ]:
# 載入服務啟動與健康檢查工具。
import os
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

# 確認目前在 Colab 且已安裝 Ollama。
if not Path("/content").is_dir():
    raise RuntimeError("找不到 /content，請先在 VS Code 選擇 Colab Kernel。")
if shutil.which("ollama") is None:
    raise RuntimeError("找不到 ollama，請先完成第 1 步。")

# 設定要傳入 Ollama 服務的環境變數。
MODEL = "nemotron-3.5-bolt:e4b"
LOCAL_API = "http://127.0.0.1:11434"
ollama_env = os.environ.copy()
ollama_env.update({
    "OLLAMA_CONTEXT_LENGTH": "65536",
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_ORIGINS": "*"
})

# 先停止舊服務，以免沿用舊的 Context 設定。
subprocess.run(["pkill", "-x", "ollama"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 將背景服務記錄到檔案，避免輸出大量雜訊。
with open("/tmp/ollama.log", "w") as log:
    ollama_proc = subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

# 至少等待三秒，並確認 Ollama API 已就緒。
time.sleep(3)
for _ in range(20):
    try:
        with urllib.request.urlopen(LOCAL_API + "/api/tags", timeout=2):
            break
    except OSError:
        time.sleep(1)
else:
    logs = Path("/tmp/ollama.log").read_text(errors="replace")[-3000:]
    raise RuntimeError("Ollama 啟動失敗：\n" + logs)

print("Ollama 已啟動：", LOCAL_API)
print("Context：65536｜Keep Alive：-1（已傳入服務程序）")

## 3. 下載模型

會直接顯示下載進度；模型已存在時，Ollama 會檢查並沿用已有的檔案。

In [ ]:
%%bash
set -euo pipefail
# 下載指定模型並顯示模型清單。
ollama pull nemotron-3.5-bolt:e4b
ollama list

## 4. 測試模型

直接顯示模型的繁體中文回覆，以及 `ollama ps` 中實際的 Context、處理器與載入情況。

In [ ]:
%%bash
set -euo pipefail
# 測試模型回覆，並確認模型載入狀態。
ollama run nemotron-3.5-bolt:e4b "(繁體中文) 你好~~~"
ollama ps

## 5. 安裝 ngrok

使用官方套件來源安裝 ngrok，顯示安裝過程與版本資訊。

In [ ]:
%%bash
set -euo pipefail
# 加入 ngrok 的套件簽章與安裝來源。
curl -fsSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
echo "deb https://ngrok-agent.s3.amazonaws.com bookworm main" | tee /etc/apt/sources.list.d/ngrok.list
# 安裝 ngrok 並顯示版本。
apt-get update -qq
apt-get install -y ngrok
ngrok version

## 6. 輸入並設定 ngrok Authtoken

執行這一格時，Notebook 會顯示**遮罩輸入欄位**。貼上 [ngrok Authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) 後按 Enter。

程式會立即執行 `ngrok config add-authtoken`，將 Token 寫入 ngrok 設定，並讓後續啟動的 ngrok 程序取得相同 Token。**不要把 Token 直接寫在 Notebook 裡。**

In [ ]:
# 使用遮罩欄位讀取 Token，避免顯示在輸出或程式碼中。
from getpass import getpass

token = getpass("請輸入 ngrok Authtoken：").strip()
if not token:
    raise ValueError("尚未輸入 Authtoken，請重新執行本格。")

# 寫入 ngrok 設定；只有憑證設定步驟隱藏指令輸出。
result = subprocess.run(["ngrok", "config", "add-authtoken", token], capture_output=True, text=True)
if result.returncode != 0:
    reason = (result.stderr or result.stdout).replace(token, "[已隱藏]")
    raise RuntimeError("ngrok Authtoken 設定失敗：" + reason)

# 同步提供給 ngrok 背景程序，避免後續連線時缺少 Token。
os.environ["NGROK_AUTHTOKEN"] = token
del token

# 確認 ngrok 設定檔格式正確，不顯示任何憑證。
result = subprocess.run(["ngrok", "config", "check"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("ngrok 設定檢查失敗：" + (result.stderr or result.stdout))
print("ngrok Authtoken 已儲存，且設定檢查通過。")

## 7. 建立 Traffic Policy

只建立您原先使用的 `/content/ollama-ngrok.yaml`，沒有 Basic Auth、密碼或安全模式選單。

In [ ]:
# 建立原本的 Traffic Policy，僅修正 Host 標頭。
POLICY = Path("/content/ollama-ngrok.yaml")
POLICY.write_text("""on_http_request:
  - actions:
      - type: add-headers
        config:
          headers:
            host: localhost
""", encoding="utf-8")
print("已建立：", POLICY)
print(POLICY.read_text(encoding="utf-8"))

## 8. 啟動 ngrok 並顯示 HTTPS URL

先檢查 Ollama、Authtoken 與 Traffic Policy 是否已準備完成，再啟動 ngrok。若 Token 無效、帳戶有限制或指令失敗，**直接顯示 `/tmp/ngrok.log` 中的錯誤內容**，而不只顯示「ngrok 已結束」。

In [ ]:
# 啟動前檢查，避免缺少前置設定卻直接啟動 ngrok。
import json

if shutil.which("ngrok") is None:
    raise RuntimeError("尚未安裝 ngrok，請先完成第 5 步。")
if not os.environ.get("NGROK_AUTHTOKEN"):
    raise RuntimeError("尚未設定 ngrok Authtoken，請先執行第 6 步。")
if not Path("/content/ollama-ngrok.yaml").is_file():
    raise RuntimeError("找不到 Traffic Policy，請先執行第 7 步。")
try:
    with urllib.request.urlopen(LOCAL_API + "/api/tags", timeout=3):
        pass
except OSError as error:
    raise RuntimeError("Ollama 服務尚未就緒，請先重新執行第 2 步。") from error

# 關閉前一次 ngrok 通道，避免重複執行時使用到舊網址。
subprocess.run(["pkill", "-x", "ngrok"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 背景啟動 ngrok，保留實際錯誤紀錄。
ngrok_log = Path("/tmp/ngrok.log")
with ngrok_log.open("w") as log:
    ngrok_proc = subprocess.Popen(
        ["ngrok", "http", "11434", "--traffic-policy-file", "/content/ollama-ngrok.yaml"],
        env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )

# 等待本機檢視介面提供對外的 HTTPS URL。
time.sleep(3)
NGROK_URL = ""
for _ in range(20):
    if ngrok_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:4040/api/tunnels", timeout=2) as response:
            tunnels = json.load(response).get("tunnels", [])
        NGROK_URL = next((t["public_url"] for t in tunnels if t.get("public_url", "").startswith("https://")), "")
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

# 啟動失敗時，直接顯示最近的 ngrok 原始錯誤訊息。
if not NGROK_URL:
    logs = ngrok_log.read_text(errors="replace")[-4000:] or "（ngrok 沒有產生日誌）"
    raise RuntimeError("ngrok 無法建立通道，請檢查下列實際錯誤：\n" + logs)

print("ngrok 已啟動。")
print("Ollama URL：", NGROK_URL)
print("OpenAI 相容 URL：", NGROK_URL + "/v1")

## 9. 測試外部網址

透過 ngrok 存取 `/api/tags`，確認對外網址實際連到 Ollama。完成後，可以將**不含 `/v1`** 的根網址提供給使用原生 Ollama API 的工具；OpenAI 相容客戶端則依需要使用 `/v1`。

In [ ]:
# 查詢遠端模型清單，確認 ngrok 已連到 Ollama。
import urllib.error

if not globals().get("NGROK_URL"):
    raise RuntimeError("尚未取得 ngrok URL，請先完成第 8 步。")

request = urllib.request.Request(NGROK_URL + "/api/tags", headers={"ngrok-skip-browser-warning": "true"})
try:
    with urllib.request.urlopen(request, timeout=30) as response:
        models = json.load(response).get("models", [])
except urllib.error.HTTPError as error:
    reason = error.read(1000).decode("utf-8", errors="replace")
    raise RuntimeError(f"遠端測試失敗（HTTP {error.code}）：\n{reason}") from None
except OSError as error:
    raise RuntimeError(f"遠端連線失敗：{error}") from None

print("遠端連線成功，模型清單：")
for model in models:
    print("-", model.get("name", "未知模型"))

## 10. 選用：卸載模型與停止服務

以下預設都不執行。需要停止時，將對應變數改為 `True`，**只重新執行這一格**。停止後原本的 ngrok 網址將不再提供服務。

In [ ]:
# 需要停止時，將對應選項改為 True。
STOP_NGROK = False
UNLOAD_MODEL = False
STOP_OLLAMA = False

# 關閉 ngrok 對外通道。
if STOP_NGROK:
    subprocess.run(["pkill", "-x", "ngrok"], check=False)
    print("ngrok 已停止。")

# 卸載模型，釋放使用中的記憶體。
if UNLOAD_MODEL:
    subprocess.run(["ollama", "stop", MODEL], check=False)
    print("已要求卸載模型。")

# 關閉 Ollama 服務。
if STOP_OLLAMA:
    subprocess.run(["pkill", "-x", "ollama"], check=False)
    print("Ollama 已停止。")

## 官方參考資料

- [Ollama Linux 安裝](https://docs.ollama.com/linux)
- [Ollama Context 與 Keep Alive](https://docs.ollama.com/faq)
- [Nemotron 3.5 Bolt E4B](https://ollama.com/library/nemotron-3.5-bolt:e4b)
- [ngrok 官方 Ollama 範例與原始 Traffic Policy](https://ngrok.com/docs/universal-gateway/examples/ollama)
- [ngrok Authtoken 設定](https://dashboard.ngrok.com/get-started/your-authtoken)
- [Google Colab 常見問題](https://research.google.com/colaboratory/intl/zh-TW/faq.html)

**驗證範圍：** 已檢查 Notebook 結構與程式語法；模型下載、GPU 推論及 ngrok 帳戶連線需要在實際的 Colab Kernel 執行確認。